# Bayesian Spatial Models: A Pedagogical Walkthrough

This notebook demonstrates all five models currently implemented in `bayespecon`:

1. SLX
2. SAR
3. SEM
4. SDM
5. SDEM

Each section explains the model idea, fits the model, and inspects posterior summaries and spatial effects.

## Conceptual Roadmap

Let $W$ denote the spatial weights matrix and $WX$ the spatial lag of covariates.

- **SLX**: $y = X\beta + WX\theta + \epsilon$
- **SAR**: $y = \rho Wy + X\beta + \epsilon$
- **SEM**: $y = X\beta + u$, $u = \lambda Wu + \epsilon$
- **SDM**: $y = \rho Wy + X\beta + WX\theta + \epsilon$
- **SDEM**: $y = X\beta + WX\theta + u$, $u = \lambda Wu + \epsilon$

In `bayespecon`, all models accept either:
- formula mode: `model_class(formula=..., data=..., W=...)`, or
- matrix mode: `model_class(y=..., X=..., W=...)`.

In [ ]:
import arviz as az
import geodatasets
import geopandas as gpd
import numpy as np
import pandas as pd
from IPython.display import display
from libpysal.graph import Graph

from bayespecon import SLX, SAR, SEM, SDM, SDEM

az.style.use("arviz-white")

In [ ]:
# Load sample spatial dataset
gdf = gpd.read_file(geodatasets.data.geoda.airbnb.url)
xcols = ["poverty", "rev_rating", "num_spots", "crowded"]
ycol = "price_pp"
gdf = gdf.dropna(subset=xcols + [ycol]).copy()

# Build contiguity graph (queen)
W = Graph.build_contiguity(gdf, rook=False).transform('r')

print(f"Observations: {len(gdf)}")
print(f"Predictors: {xcols}")

## Helper Functions

To keep each model section focused, we use utility functions to:
- fit with consistent MCMC settings,
- print posterior summaries,
- tabulate direct/indirect/total effects.

For a quick pedagogical run, we use small draw counts. Increase these for real inference.

In [ ]:
def fit_and_report(model_cls, formula, data, W, draws=400, tune=400, chains=4, seed=42):
    model = model_cls(
        formula=formula,
        data=data,
        W=W,
        logdet_method="eigenvalue",
    )

    idata = model.fit(
        draws=draws,
        tune=tune,
        chains=chains,
        target_accept=0.9,
        random_seed=seed,
        progressbar=False,
        idata_kwargs={"log_likelihood": True},
    )

    summary = model.summary(round_to=3)
    effects = model.spatial_effects()
    effects_df = pd.DataFrame({
        "feature": effects["feature_names"],
        "direct": effects["direct"],
        "indirect": effects["indirect"],
        "total": effects["total"],
    })

    return model, idata, summary, effects_df

## Convergence Diagnostics Helper

For each fitted model, we will inspect:
- `r_hat` (target close to 1.00)
- effective sample sizes (`ess_bulk`, `ess_tail`)
- trace plots for key parameters.

In [ ]:
import matplotlib.pyplot as plt


def diagnostics_table(idata, var_names):
    cols = ["mean", "sd", "ess_bulk", "ess_tail", "r_hat"]
    return az.summary(idata, var_names=var_names, round_to=3)[cols]


def show_trace(idata, var_names, title):
    az.plot_trace(idata, var_names=var_names)
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

## 1) SLX: Spatially Lagged Covariates Only

Model:

$$
y = X\beta + WX\theta + \epsilon
$$

Interpretation:
- `beta` captures local covariate effects.
- `theta` captures neighbor-covariate spillovers.
- No spatial lag on $y$, so no autoregressive propagation through outcomes.

In [ ]:
slx, idata_slx, summary_slx, effects_slx = fit_and_report(
    SLX,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
display(summary_slx)
display(effects_slx)


In [ ]:
az.plot_forest(idata_slx)
display(diagnostics_table(idata_slx, ["beta", "sigma"]))
show_trace(idata_slx, ["sigma"], "SLX Trace: sigma")

## 2) SAR: Spatial Lag of Outcome

Model:

$$
y = \rho Wy + X\beta + \epsilon
$$

Interpretation:
- $\rho$ controls feedback from neighboring outcomes.
- Effects are amplified through $(I - \rho W)^{-1}$, so direct and indirect impacts differ from raw $\beta$.

In [ ]:
sar, idata_sar, summary_sar, effects_sar = fit_and_report(
    SAR,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
display(summary_sar)
display(effects_sar)

In [ ]:
az.plot_forest(idata_sar)
display(diagnostics_table(idata_sar, ["rho", "beta", "sigma"]))
show_trace(idata_sar, ["rho", "sigma"], "SAR Trace: rho, sigma")

## 3) SEM: Spatial Correlation in Errors

Model:

$$
y = X\beta + u, \quad u = \lambda Wu + \epsilon
$$

Interpretation:
- Spatial dependence is moved to latent shocks rather than outcome feedback.
- Useful when omitted spatial factors induce correlated residuals.

In [ ]:
sem, idata_sem, summary_sem, effects_sem = fit_and_report(
    SEM,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
display(summary_sem)
display(effects_sem)

In [ ]:
display(diagnostics_table(idata_sem, ["lam", "beta", "sigma"]))
show_trace(idata_sem, ["lam", "sigma"], "SEM Trace: lam, sigma")

## 4) SDM: SAR + SLX

Model:

$$
y = \rho Wy + X\beta + WX\theta + \epsilon
$$

Interpretation:
- Includes both outcome feedback and neighbor-covariate channels.
- Often used as a flexible nesting model for SAR/SLX.

In [ ]:
sdm, idata_sdm, summary_sdm, effects_sdm = fit_and_report(
    SDM,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
display(summary_sdm)
display(effects_sdm)

In [ ]:
az.plot_forest(idata_sdm)
display(diagnostics_table(idata_sdm, ["rho", "beta", "sigma"]))
show_trace(idata_sdm, ["rho", "sigma"], "SDM Trace: rho, sigma")

## 5) SDEM: SLX + Spatial Error

Model:

$$
y = X\beta + WX\theta + u, \quad u = \lambda Wu + \epsilon
$$

Interpretation:
- Neighbor covariates matter directly (through $WX$),
- and unmodeled shocks are spatially correlated (through $u$).

In [ ]:
sdem, idata_sdem, summary_sdem, effects_sdem = fit_and_report(
    SDEM,
    formula="price_pp ~ poverty + rev_rating + num_spots + crowded",
    data=gdf,
    W=W,
)
display(summary_sdem)
display(effects_sdem)

In [ ]:
display(diagnostics_table(idata_sdem, ["lam", "beta", "sigma"]))
show_trace(idata_sdem, ["lam", "sigma"], "SDEM Trace: lam, sigma")

## Spatial Specification Diagnostics (New API)

The model classes now expose dedicated specification diagnostics that return a structured `DiagnosticResult` object.

Below we run each model's `spatial_specification_tests()` battery and tabulate:
- test name
- statistic
- p-value
- any extra test-specific metadata

In [ ]:
def diagnostic_to_row(result):
    row = {
        "diagnostic": result.name,
        "statistic": float(result.statistic),
        "pvalue": float(result.pvalue),
    }
    if result.extra:
        for k, v in result.extra.items():
            row[k] = float(v) if isinstance(v, (int, float, np.number)) else v
    return row


spec_models = {
    "SLX": slx,
    "SAR": sar,
    "SEM": sem,
    "SDM": sdm,
    "SDEM": sdem,
}

spec_tables = {}
for model_name, model_obj in spec_models.items():
    tests = model_obj.spatial_specification_tests()
    table = pd.DataFrame([diagnostic_to_row(dr) for dr in tests.values()])
    spec_tables[model_name] = table.sort_values("diagnostic").reset_index(drop=True)

for model_name, table in spec_tables.items():
    print(model_name)
    display(table)
    print()

## Compare Spatial Parameters Across Models

This cell compares posterior means/intervals for the spatial scalar where present:
- `rho` in SAR/SDM
- `lam` in SEM/SDEM

In [ ]:
def pull_scalar(idata, var_name):
    if var_name not in idata.posterior:
        return None
    vals = idata.posterior[var_name].values.reshape(-1)
    return {
        "mean": float(np.mean(vals)),
        "sd": float(np.std(vals)),
        "hdi_3%": float(np.quantile(vals, 0.03)),
        "hdi_97%": float(np.quantile(vals, 0.97)),
    }

rows = []
for name, idata, var in [
    ("SAR", idata_sar, "rho"),
    ("SEM", idata_sem, "lam"),
    ("SDM", idata_sdm, "rho"),
    ("SDEM", idata_sdem, "lam"),
]:
    stats = pull_scalar(idata, var)
    stats["model"] = name
    stats["parameter"] = var
    rows.append(stats)

spatial_param_table = pd.DataFrame(rows)[["model", "parameter", "mean", "sd", "hdi_3%", "hdi_97%"]]
display(spatial_param_table)

## Model Comparison: WAIC and LOO

We compare fitted models using information criteria from ArviZ.
Lower values are better on these scales.

Notes:
- These comparisons are only meaningful when models are fit to the same data.
- For robust results, increase `draws` and `tune` beyond this pedagogical setup.
- Some model formulations may not expose pointwise `log_likelihood`; those are skipped in the joint comparison and still reported in per-model attempts.

In [ ]:
idata_dict = {
    "SLX": idata_slx,
    "SAR": idata_sar,
    "SEM": idata_sem,
    "SDM": idata_sdm,
    "SDEM": idata_sdem,
}


def _has_loglik(idata):
    if "log_likelihood" not in idata.groups():
        return False
    return len(list(idata.log_likelihood.data_vars)) > 0


valid_for_compare = {name: idata for name, idata in idata_dict.items() if _has_loglik(idata)}
missing_loglik = [name for name, idata in idata_dict.items() if not _has_loglik(idata)]

if missing_loglik:
    print("Skipped (no log_likelihood group):", ", ".join(missing_loglik))

if len(valid_for_compare) < 2:
    print("Need at least two models with valid log_likelihood for WAIC/LOO compare.")
else:
    for ic in ("waic", "loo"):
        try:
            cmp = az.compare(valid_for_compare, ic=ic, method="BB-pseudo-BMA")
            print(f"{ic.upper()} comparison")
            display(cmp)
        except Exception as e:
            print(f"{ic.upper()} comparison not available: {type(e).__name__}: {e}")

# Always provide per-model IC attempts so the cell remains informative.
rows = []
for name, idata in idata_dict.items():
    row = {"model": name}

    try:
        waic_res = az.waic(idata)
        row["elpd_waic"] = float(waic_res.elpd_waic)
        row["p_waic"] = float(waic_res.p_waic)
    except Exception as e:
        row["elpd_waic"] = np.nan
        row["p_waic"] = np.nan
        row["waic_error"] = str(e).split("\n", 1)[0]

    try:
        loo_res = az.loo(idata)
        row["elpd_loo"] = float(loo_res.elpd_loo)
        row["p_loo"] = float(loo_res.p_loo)
    except Exception as e:
        row["elpd_loo"] = np.nan
        row["p_loo"] = np.nan
        row["loo_error"] = str(e).split("\n", 1)[0]

    rows.append(row)

ic_table = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
display(ic_table)

## MATLAB-Style Residual Diagnostics (Per Model)

This section mirrors the MATLAB reference diagnostics module:
- `rdiagnose`-style influence and outlier diagnostics
- Breusch-Pagan heteroskedasticity test (`bpagan`)
- ARCH test (`arch`)
- Ljung-Box residual autocorrelation test (`qstat2`)

Interpretation quick guide:
- Smaller p-values for BPagan/ARCH/Ljung-Box suggest diagnostics concerns.
- Outlier counters are threshold-based candidate flags, not automatic exclusions.

In [ ]:
pd.DataFrame(slx.spatial_effects())

In [ ]:
models = {
    "SLX": slx,
    "SAR": sar,
    "SEM": sem,
    "SDM": sdm,
    "SDEM": sdem,
}


def _last_value(x):
    arr = np.asarray(x)
    if arr.size == 0:
        return np.nan
    return float(arr.reshape(-1)[-1])


rows = []
for name, m in models.items():
    try:
        het = m.heteroskedasticity_diagnostics(arch_lags=[1, 2, 4])
        ac = m.autocorrelation_diagnostics(lags=[1, 2, 4])
        out = m.outlier_diagnostics()

        bp = het["bpagan"]
        arch = het["arch"]

        # Keep one compact row per model for quick pedagogical comparison.
        rows.append({
            "model": name,
            "bpagan_lm": float(bp.statistic),
            "bpagan_p": float(bp.pvalue),
            "arch_lag4_stat": _last_value(arch.statistic),
            "arch_lag4_p": _last_value(arch.pvalue),
            "lb_lag4_q": _last_value(ac.statistic),
            "lb_lag4_p": _last_value(ac.pvalue),
            "n_leverage_flags": int(len(out["leverage_idx"])),
            "n_rstudent_flags": int(len(out["rstudent_idx"])),
            "n_dffit_flags": int(len(out["dffit_idx"])),
            "n_dfbeta_flags": int(out["dfbeta_idx"].shape[0]),
        })
    except Exception as e:
        print(f"{name} failed: {type(e).__name__}: {e}")

if rows:
    diag_summary = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
else:
    diag_summary = pd.DataFrame(columns=[
        "model",
        "bpagan_lm",
        "bpagan_p",
        "arch_lag4_stat",
        "arch_lag4_p",
        "lb_lag4_q",
        "lb_lag4_p",
        "n_leverage_flags",
        "n_rstudent_flags",
        "n_dffit_flags",
        "n_dfbeta_flags",
    ])

display(diag_summary)

# Optional deep dive for one model
example = "SAR"
rd = models[example].regression_diagnostics()
print(f"Detailed rdiagnose-style outputs available for {example}:", sorted(rd.keys()))

In [ ]:
het_sar = sar.heteroskedasticity_diagnostics(arch_lags=[1, 2, 4])

sar_het_table = pd.DataFrame([
    {
        "diagnostic": "bpagan",
        "statistic": float(het_sar["bpagan"].statistic),
        "pvalue": float(het_sar["bpagan"].pvalue),
    },
    {
        "diagnostic": "arch_lag4",
        "statistic": _last_value(het_sar["arch"].statistic),
        "pvalue": _last_value(het_sar["arch"].pvalue),
    },
])

sar_het_table

## Matrix-Mode API Example (Optional)

The package also supports direct matrix inputs. This is useful if your design matrix is already engineered.

In [ ]:
y = gdf[ycol]
X = gdf[xcols]

sar_matrix = SAR(y=y, X=X, W=W)
idata_sar_matrix = sar_matrix.fit(draws=200, tune=200, chains=2, random_seed=7, progressbar=False)
display(sar_matrix.summary(round_to=3))

## Notes for Real Analyses

- Increase `draws` and `tune` substantially (for example, 2000+ each).
- Inspect convergence diagnostics (`r_hat`, ESS, trace plots).
- Compare models with posterior predictive checks and information criteria.
- Consider prior sensitivity checks for spatial parameters (`rho`, `lam`).